# Day 37: Text‑to‑3D with Shap‑E

Generate a 3D object from a text prompt.

In [ ]:
import torch
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_mesh, display_3d_volume, decode_latent_images
import plotly.graph_objects as go

In [ ]:
# Load models (this may take a few minutes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
xm = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))

In [ ]:
# Generate 3D latents from text
prompt = "a red apple"
batch_size = 1
guidance_scale = 15.0

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(texts=[prompt] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)
print("Latents generated")

In [ ]:
# Decode and save mesh
for i, latent in enumerate(latents):
    mesh = decode_latent_mesh(xm, latent).tri_mesh()
    with open(f"output_{i}.obj", "w") as f:
        mesh.write_obj(f)
    print(f"Saved output_{i}.obj")

In [ ]:
# Visualise using Plotly (requires trimesh installed)
import trimesh
mesh = trimesh.load("output_0.obj")
scene = mesh.as_geometry()
scene.show()  # Opens an interactive window

## Alternative: Using Shap‑E via CLI (if notebook not working)
```bash
python -m shap_e.diffusion.sample --prompt "a red apple" --output_dir ./output
```